# OSS Console — Train the **SAR** base model on PUBLIC data

Trains a person-in-water / boat / life-raft detector from **SeaDronesSee (CC0 — commercially safe)** — **no manual labeling needed**. Exports **ONNX** to drop into the app.

**Runtime → Change runtime type → GPU (T4)** before running. Plan: `docs/MODEL-TRAINING.md`.


In [ ]:
# 1. Install
!pip -q install ultralytics onnx onnxruntime pyyaml

## 2. Get SeaDronesSee (CC0)
1. Register (free) at **https://www.macvi.org/dataset** and download the **Object Detection v2** set (images + COCO annotations).
2. Upload the zip(s) here via the **Files** panel (or mount Google Drive), then set the path below.

⚠️ Use **SeaDronesSee only** for weights you ship commercially. **Do NOT add AFO** (non-commercial license).


In [ ]:
import os, glob, zipfile
os.makedirs('/content/sds', exist_ok=True)
for z in glob.glob('/content/*.zip'):
    with zipfile.ZipFile(z) as f: f.extractall('/content/sds')
print('contents:', os.listdir('/content/sds'))
# Find the COCO annotation json (train) and the images dir — adjust if your layout differs:
ANN = glob.glob('/content/sds/**/*train*.json', recursive=True)
print('annotation candidates:', ANN)

## 3. COCO → YOLO + auto-build data.yaml
Class names are read straight from the dataset's own categories — no hardcoding/guessing.


In [ ]:
import json, yaml
from ultralytics.data.converter import convert_coco
ann_dir = os.path.dirname(ANN[0])           # folder holding the COCO json(s)
convert_coco(labels_dir=ann_dir, save_dir='/content/sds_yolo', use_segments=False)
# derive class names in id order from the COCO json
j = json.load(open(ANN[0]))
cats = sorted(j['categories'], key=lambda c: c['id'])
names = {i:c['name'] for i,c in enumerate(cats)}
print('classes:', names)
# NOTE: point train/val image dirs at your unzipped SeaDronesSee images (edit if needed):
data = {'path':'/content/sds_yolo', 'train':'images/train2017', 'val':'images/val2017', 'names':names}
open('/content/sar.yaml','w').write(yaml.safe_dump(data))
print(open('/content/sar.yaml').read())

### (alternative) Roboflow Universe maritime/SAR dataset
If you'd rather use a one-click Roboflow set, paste its download snippet here (free Roboflow key) and point `sar.yaml` at the exported `data.yaml`.


In [ ]:
# from roboflow import Roboflow
# rf = Roboflow(api_key='YOUR_KEY'); ds = rf.workspace('...').project('...').version(N).download('yolov11')
# then set sar.yaml path to ds.location + '/data.yaml'
print('optional — fill in your Roboflow snippet if using this path')

## 4. Train YOLO11 @ 1280 (small top-down targets + water/glare aug)


In [ ]:
from ultralytics import YOLO
m = YOLO('yolo11n.pt')   # n=fast start; s/m for more accuracy
m.train(data='/content/sar.yaml', epochs=120, imgsz=1280, batch=8, name='oss-sar',
        patience=30, hsv_h=0.015, hsv_s=0.7, hsv_v=0.5, fliplr=0.5, mosaic=1.0, degrees=10)

### (optional) benchmark YOLOv8 — often better on tiny aerial targets
Train both, keep whichever wins val mAP on this data.


In [ ]:
# YOLO('yolov8n.pt').train(data='/content/sar.yaml', epochs=120, imgsz=1280, batch=8, name='oss-sar-v8')

## 5. Export ONNX (browser-ready) → send it over


In [ ]:
best='runs/detect/oss-sar/weights/best.pt'
YOLO(best).export(format='onnx', imgsz=1280, simplify=True)   # add int8=True for a smaller phone model
print('ONNX ->', best.replace('.pt','.onnx'))
# Download best.onnx from the Files panel and send it to wire into the SAR edition.

## 6. Wire into the app
Send me `best.onnx` + the class list and I'll load it into the SAR edition's neural-detection slot (ONNX Runtime Web), and flip the loop panel's **Deploy** stage to active.

_Then improve it: add your own labeled drone frames (uncertainty-first) and retrain — the loop sharpens with every pass._
